# location-maison-model-for-google-collab - GitHub + Drive

Notebook Colab pour :
- cloner le code depuis GitHub
- monter Google Drive
- rediriger les artefacts vers Drive
- générer le dataset
- entraîner le modèle
- évaluer et tester le modèle


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Configuration

Renseigne ici l'URL GitHub et le dossier Drive cible.


In [ ]:
GIT_REPO_URL = 'https://github.com/TON-COMPTE/location-maison-model-for-google-collab.git'
GIT_BRANCH = 'main'
PROJECT_NAME = 'location-maison-model-for-google-collab'
DRIVE_ROOT = f'/content/drive/MyDrive/IA/{PROJECT_NAME}'
WORKTREE = f'/content/{PROJECT_NAME}'

print('GIT_REPO_URL =', GIT_REPO_URL)
print('GIT_BRANCH   =', GIT_BRANCH)
print('DRIVE_ROOT   =', DRIVE_ROOT)
print('WORKTREE     =', WORKTREE)


In [ ]:
import os

for path in [
    f'{DRIVE_ROOT}/outputs/checkpoints',
    f'{DRIVE_ROOT}/outputs/metrics',
    f'{DRIVE_ROOT}/outputs/reports',
    f'{DRIVE_ROOT}/logs/app',
    f'{DRIVE_ROOT}/logs/train',
    f'{DRIVE_ROOT}/logs/error',
    f'{DRIVE_ROOT}/data/datasets',
    f'{DRIVE_ROOT}/data/smoke_datasets',
    f'{DRIVE_ROOT}/data/processed/relevance',
]:
    os.makedirs(path, exist_ok=True)

print('Drive layout ready')


## Mise a jour du code sans toucher aux artefacts Drive

- Cette cellule met a jour le repo dans `/content` si le code existe deja.
- Les modeles, checkpoints, logs et rapports restent dans Drive via les liens symboliques.
- Donc une mise a jour du code ne supprime pas les modeles stockes dans Drive.


In [ ]:
import os

if os.path.isdir(WORKTREE):
    print('Repo deja present, mise a jour du code en cours...')
    %cd {WORKTREE}
    !git fetch origin
    !git checkout {GIT_BRANCH}
    !git pull --ff-only origin {GIT_BRANCH}
else:
    print('Clone initial du repo...')
    %cd /content
    !git clone --branch {GIT_BRANCH} {GIT_REPO_URL} {WORKTREE}
    %cd {WORKTREE}

!pwd
!git rev-parse --short HEAD
!ls


## Rediriger les artefacts vers Drive

Le code reste dans `/content`, mais les artefacts sont stockés dans Drive.


In [ ]:
%cd {WORKTREE}
# On recree seulement les liens symboliques locaux; les artefacts dans Drive ne sont pas supprimes.
!rm -rf outputs logs data/datasets data/smoke_datasets data/processed
!mkdir -p data
!ln -s {DRIVE_ROOT}/outputs outputs
!ln -s {DRIVE_ROOT}/logs logs
!ln -s {DRIVE_ROOT}/data/datasets data/datasets
!ln -s {DRIVE_ROOT}/data/smoke_datasets data/smoke_datasets
!ln -s {DRIVE_ROOT}/data/processed data/processed
!ls -l
!ls -l data


In [ ]:
import torch
print('CUDA dispo :', torch.cuda.is_available())
print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'aucun')


In [ ]:
%cd {WORKTREE}
!pip install -r requirements.txt


## Optionnel : token Hugging Face


In [ ]:
import os
HF_TOKEN = ''
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOKEN
    print('HF token configured')
else:
    print('HF token not set')


## Mettre à jour le code depuis GitHub

Relance cette cellule si tu pousses une nouvelle version sur GitHub.


In [ ]:
%cd {WORKTREE}
!git fetch --all
!git checkout {GIT_BRANCH}
!git pull --ff-only origin {GIT_BRANCH}


## Préparer les sources et générer le dataset


In [ ]:
%cd {WORKTREE}
!python scripts/dataset/prepare_sources.py
!PYTHONPATH=src python -m location_maison_model_annonce.cli.generate_dataset --config config/project.yaml
!cat data/datasets/dataset_manifest.json


## Entraînement complet


In [ ]:
%cd {WORKTREE}
!PYTHONPATH=src python -m location_maison_model_annonce.cli.train --config config/project.yaml


## Évaluation seule avec reprise incrémentale

- Si une évaluation précédente a été interrompue, le script reprend depuis `outputs/reports/*.partial.jsonl`.
- Vérifie d'abord l'état et les fichiers partiels, puis relance la commande d'évaluation.


In [ ]:
%cd {WORKTREE}
!PYTHONPATH=src python -m location_maison_model_annonce.cli.status --config config/project.yaml
!ls -lh outputs/reports
!PYTHONPATH=src python -m location_maison_model_annonce.cli.evaluate --config config/project.yaml --split test


## Résultats


In [ ]:
%cd {WORKTREE}
!cat outputs/metrics/latest_metrics.json
!ls outputs/checkpoints
!ls outputs/reports


## Test du modèle


In [ ]:
%cd {WORKTREE}
!PYTHONPATH=src python -m location_maison_model_annonce.cli.predict --config config/project.yaml --max-new-tokens 320 --text "Studio à louer 2 chambres douche cuisine wc buanderie dans la barrière avec la grille de sécurité avec l'eau à 120000 f à Okolassi contact 066177172"


## Logs utiles

- `RUN_STATE.json` permet de voir immédiatement si le train ou l'évaluation a repris correctement.


In [ ]:
%cd {WORKTREE}
!cat outputs/RUN_STATE.json
!tail -n 100 logs/train/train.log
!tail -n 100 logs/app/application.log
